## 02 — The Comparison

This is the final notebook.

We put both approaches side by side: the system we built across seven modules and the one command that replaces most of it. The goal is not to show that `tippecanoe` is better — it obviously is for production use — but to show exactly **what work it is doing**, because we built every one of those pieces ourselves.

## System Comparison — Architecture

```
OUR SYSTEM                                  TIPPECANOE + TILE CLIENT
─────────────────────────────────────       ──────────────────────────────────────
ne_10m_railroads.geojson (40 MB)            ne_10m_railroads.geojson (40 MB)
       │                                           │
       ▼                                           ▼
Module 02: simplify at 4 epsilons           tippecanoe (one command, ~30 seconds)
  → 4 GeoJSON output files                        │
       │                                           ▼
       ▼                                    railroads.pmtiles (~3 MB)
Module 04: build 4 GridIndex objects               │
  (startup: ~5 seconds)                            ▼
       │                               tile client (browser or localtileserver)
       ▼                                 fetches only visible tiles on demand
Module 05: get_lod(zoom) selects index
       │
       ▼
Module 03: bbox cull within selected index
       │
       ▼
GeoJSON layer (re-sent every pan/zoom)
```

## System Comparison — Numbers

In [ ]:
from pathlib import Path

lod_dir = Path("../../data/lod")
raw     = Path("../../data/ne_10m_railroads.geojson")
pmtiles = Path("../../data/railroads.pmtiles")

lod_files = [
    "railroads_coarse.geojson",
    "railroads_medium.geojson",
    "railroads_fine.geojson",
    "railroads_extra_fine.geojson",
]

our_total_mb = sum((lod_dir / f).stat().st_size for f in lod_files) / 1_000_000
raw_mb       = raw.stat().st_size / 1_000_000
pm_mb        = pmtiles.stat().st_size / 1_000_000 if pmtiles.exists() else None

print("Storage comparison:")
print(f"  Raw GeoJSON:               {raw_mb:.1f} MB")
print(f"  Our 4 LOD files (total):   {our_total_mb:.1f} MB")
if pm_mb:
    print(f"  tippecanoe PMTiles:        {pm_mb:.1f} MB")
else:
    print("  tippecanoe PMTiles:        (run Notebook 01 first)")

print()
print("Runtime comparison:")
print(f"  Our startup (load + index): ~5–10s")
print(f"  Our per-query time:         ~1–5ms")
print(f"  Tile client startup:        ~0s (lazy fetch)")
print(f"  Tile fetch (per tile):      ~10–50ms over network")
print(f"  Tile fetch (local):         <1ms")

## What Tippecanoe Automated — Specifically

Now name each thing precisely — because you built it:

**1. Multi-resolution simplification (our Module 02)**
tippecanoe applies a tolerance appropriate for each zoom level automatically. You do not choose 4 epsilons — you choose one `--simplification` value and it scales it per zoom.

**2. Spatial bucketing (our Module 04)**
Tiles are the index. There is no separate grid index to build — the tile `(z, x, y)` address is the bucket. Features are pre-assigned to tiles at generation time.

**3. Viewport culling (our Module 03)**
The client requests only the tiles its viewport covers. No intersection test — irrelevant tiles are never fetched.

**4. LOD switching (our Module 05)**
The tile URL includes the zoom level (`/{z}/`). The client naturally requests tiles at the right zoom. No decision function needed.

**5. Binary encoding**
We never built this — our GeoJSON is text. Tippecanoe outputs MVT binary with integer coordinates. ~5× smaller and faster to parse.

**6. Streaming / lazy loading**
We never built this either. Tiles are fetched on demand; unused regions are never touched.

## What Tippecanoe Does NOT Do

Tippecanoe is a **preprocessing pipeline** — it generates static files. It does not:

- Serve tiles dynamically (you still need a static host, CDN, or `localtileserver`)
- Filter features at query time based on user input
- Handle real-time data updates
- Decide which features to show based on screen density or user preferences at render time

For live, user-driven filtering (e.g., "show only electrified railways"), a tile system either:
- Pre-generates multiple tile sets (one per filter combination)
- Sends all attributes in the tile and lets the client filter at render time (style expressions)
- Uses a dynamic tile server that queries a database per request

None of these are trivial. Our handbuilt system could add real-time filtering in ten lines.

## The Quote That Started This

> *"Build the smallest version that teaches the idea. Borrow the version that survives the real world."*

You built the smallest version. You know:
- What Douglas-Peucker does and why epsilon matters
- Why spatial indexes exist and what they trade off
- Why bounding box culling is O(n) without an index
- Why the tile coordinate scheme is itself a spatial index
- Why binary encoding is worth the complexity

When you use `tippecanoe` from now on, you can read its flags without guessing. When it produces unexpected output, you can reason about why. When a colleague says "we should just use vector tiles" without understanding the tradeoffs, you can ask the right questions.

That is not the same as having typed `tippecanoe` once.

## Exercise A

Write a one-page (or ~15 bullet point) technical comparison of the two systems. Cover:
- Build time
- Storage footprint
- Startup cost for the end user
- Per-request data transfer
- Support for real-time filtering
- Complexity to maintain
- What you would use for a class project vs. a production application

Write it in the cell below as markdown.

*Your comparison here.*

## Exercise B

Look up the `--attribute-filter` and `--include` flags in the tippecanoe documentation.

Could you use these to produce a tile set that only includes electrified railroads (`electric` property)? Write the command you would use, and explain what the resulting tile set would and would not be able to show.

In [ ]:
# Write the tippecanoe command for an electrified-only tile set as a comment
# Explain what it can and cannot show
# Your answer here

## Check Your Understanding

A classmate who skipped Modules 01–06 and came straight to this notebook could run `tippecanoe` and get a working tile set. They would see the flags but not know what they mean.

Name **three specific situations** where their lack of understanding would cost them — where they would make a wrong decision, miss a bug, or be unable to debug a problem — that you would be able to handle.

---

## End of the Data Manager Micro Lessons

You built:
- A simplification algorithm
- A multi-resolution data pipeline
- A spatial intersection test
- A grid-based spatial index
- A zoom-driven data selection system
- A working interactive map viewer
- An informed opinion about when to stop building and borrow instead

The railroad project is where you apply it.